<a href="https://colab.research.google.com/github/shahwaiz-9/Machine_Learning/blob/main/Text_preprocessing_%26_Multiclass_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 65.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 203.1 kB/s eta 0:00:00


In [30]:
from sklearn.datasets import fetch_20newsgroups

In [81]:

categories = [
    'alt.atheism',
    'comp.graphics',
    'comp.os.ms-windows.misc',
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'rec.autos',
    'rec.motorcycles',
    'sci.med',
    'sci.space',
    'soc.religion.christian'
]

train_data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)


test_data = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)



In [111]:
X_train_texts = train_data.data
X_test_texts  = test_data.data
y_train       = train_data.target
y_test        = test_data.target

In [112]:
X_train_texts[0]

"TO: saz@hook.corp.mot.com\n\n\nSZ>Does anybody know of a program that converts .GIF files to .BMP files\nSZ>and if so, where can I ftp it from?  Any help would be greatly\nSZ>appreciated.\n\n  Sure... A GREAT shareware  program is Graphic Workshop (the newest\n  version is 6.1).  Although I don't know where you can ftp it from.  It\n  also converts to about 15 other formats, and does MANY other things.\n\n....r.c V.t.ell. .r..."

In [113]:
total_words_xtrain = 0
for text in X_train_texts:
  total_words_xtrain += len(text.split())

print(f"Total words in X_train_texts: {total_words_xtrain}")

Total words in X_train_texts: 933401


we have to remove the followng by looking at the text
1. Line and space breakers \n , \t etc
2. Extra Spaces
3. Punctuations
4. Email refrences ( optional )
5. Lower case text

In [85]:
import re
import contractions

In [53]:
# def clean_text(text):

#   # Remove html tags
#   text = re.sub(r'<.*?>', '', text)

#   # Remove line breaks
#   text = re.sub(r'\n', ' ', text)

#   # Remove tab spaces

#   text = re.sub(r'\t', ' ', text)

#   # Remove Url's

#   text = re.sub(r'http\S+|www\S+|https\S+', '', text)

#   # Lower case

#   text = text.lower()

#   # Remove Punctuations

#   text = re.sub(r'[^\w\s]', '', text)

#   text = re.sub(r"[^A-Za-z0-9 ]+", " ", text)

#   # Remove Extra Spaces

#   text = re.sub(r'\s+', ' ', text)

#   return text


In [114]:
def clean_text(text):

  text = re.sub(r'<.*?>', '', text)

  text = contractions.fix(text)

    # 3. Lowercase
  text = text.lower()


  text = re.sub(r'\S*@\S*\s?', '', text)

    # 5. Remove URLs
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # 6. Remove specific 20Newsgroup noise (like "sz>" or "ax>")
  text = re.sub(r'\b[a-z]{1,2}>\s?', '', text)

    # 7. Remove Punctuations and Numbers
    # Using [^a-z\s] ensures we only keep alphabetical words
  text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Remove Extra Spaces and single-letter "residue"
  words = text.split()
    # Filtering for len > 1 removes things like "r c v t ell r" seen in your text
  clean_words = [w for w in words if len(w) > 1]

  return " ".join(clean_words)

In [115]:
X_train_texts = [clean_text(text) for text in X_train_texts]
X_test_texts = [clean_text(text) for text in X_test_texts]

In [116]:
total_words_xtrain = 0
for text in X_train_texts:
  total_words_xtrain += len(text.split())

print(f"Total words in X_train_texts: {total_words_xtrain}")

Total words in X_train_texts: 898305


In [109]:
X_train_texts[10]

'was laughing about the law part have driven thru soho manahattan know what you are talking bout not that durham nc is any better well maybe little bit anyway but the nc dot takes more money from road taxes and puts it in their own pockets and into the pockets of the guys building the large condos that need their own roads than they do back into fixing roads but hey the local paper did report of this last summer and boy am glad do not work for the dot because they got shat on bigtime wonder who lost their jobs ed got any idea'

In [90]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet

In [60]:
# Download necessary NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [119]:
english_stopwords_set = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [62]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN # Default to noun if not found

In [120]:
def tokenize_lemmitize(text):

  # Word tokenization
  tokens = word_tokenize(text)

  pos_tags = nltk.pos_tag(tokens)

  lemmatized = []

  for word, tag in pos_tags:
    wn_tag = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos=wn_tag)

    if word not in english_stopwords_set and len(lemma) > 1:
      lemmatized.append(lemma)


  return " ".join(lemmatized)

In [ ]:
X_train_texts = [tokenize_lemmitize(text) for text in X_train_texts]
X_test_texts = [tokenize_lemmitize(text) for text in X_test_texts]

In [ ]:
X_train_texts[100]

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [123]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=20000,
    ngram_range=(1, 3),
    min_df=5,
    max_df=0.7,
    sublinear_tf=True
)

# Fit the vectorizer on the training data and transform both training and test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_texts)
X_test_tfidf = tfidf_vectorizer.transform(X_test_texts)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Shape of X_train_tfidf: (5801, 11772)
Shape of X_test_tfidf: (3861, 11772)


**Modeling**

In [124]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [125]:
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"Logistic Regression Accuracy: {lr_accuracy}")

Logistic Regression Accuracy: 0.7464387464387464


In [126]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)
nb_accuracy = accuracy_score(y_test, nb_pred)
print(f"Naive Bayes Accuracy: {nb_accuracy}")

Naive Bayes Accuracy: 0.732970732970733


In [80]:
svm_model = SVC()
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)
svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"SVM Accuracy: {svm_accuracy}")

SVM Accuracy: 0.746956746956747


In [77]:
from sklearn.svm import LinearSVC


# Try LinearSVC instead of MultinomialNB
l_model = LinearSVC(C=1.0, max_iter=1000)
l_model.fit(X_train_tfidf, y_train)

y_pred = l_model.predict(X_test_tfidf)
print(f"New Accuracy: {accuracy_score(y_test, y_pred)}")

New Accuracy: 0.7464387464387464


**Hyperparameter tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Define the parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['liblinear', 'saga']
}

# Initialize Logistic Regression model
lr = LogisticRegression(max_iter=1000, random_state=42) # Increase max_iter for convergence

# Initialize GridSearchCV
grid_search_lr = GridSearchCV(lr, param_grid, cv=5, verbose=2, n_jobs=-1, scoring='accuracy')

# Fit GridSearchCV to the training data
grid_search_lr.fit(X_train_tfidf, y_train)

# Print the best parameters and best score
print(f"Best parameters for Logistic Regression: {grid_search_lr.best_params_}")
print(f"Best cross-validation accuracy for Logistic Regression: {grid_search_lr.best_score_}")

# Evaluate the best model on the test set
best_lr_model = grid_search_lr.best_estimator_
best_lr_pred = best_lr_model.predict(X_test_tfidf)
best_lr_accuracy = accuracy_score(y_test, best_lr_pred)
print(f"Tuned Logistic Regression Test Accuracy: {best_lr_accuracy}")

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best parameters for Logistic Regression: {'C': 100, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation accuracy for Logistic Regression: 0.7879666755769402
Tuned Logistic Regression Test Accuracy: 0.7446257446257446


In [ ]:
sample_doc = "This article discusses the latest advancements in space exploration, including new missions to Mars and the development of reusable rockets. It also touches on the challenges of deep space travel and the search for extraterrestrial life."

In [ ]:
# Apply cleaning function
processed_sample_doc = clean_text(sample_doc)

# Apply tokenization and lemmatization function
processed_sample_doc = tokenize_lemmitize(processed_sample_doc)

print(f"Processed Sample Document: {processed_sample_doc}")

Processed Sample Document: article discuss late advancement space exploration include new mission mar development reusable rocket also touch challenge deep space travel search extraterrestrial life


In [ ]:
sample_doc_tfidf = tfidf_vectorizer.transform([processed_sample_doc])

print(f"Shape of sample_doc_tfidf: {sample_doc_tfidf.shape}")

Shape of sample_doc_tfidf: (1, 63929)


In [ ]:
predicted_category_index = best_lr_model.predict(sample_doc_tfidf)[0]
predicted_category_name = categories[predicted_category_index]

print(f"The predicted category for the sample document is: {predicted_category_name}")

The predicted category for the sample document is: sci.space


In [39]:
def singal_word_removal(text):
  words = text.split()

  words = [w for w in words if len(w) > 1]

  return " ".join(words)


In [42]:
X_train_texts = [singal_word_removal(text) for text in X_train_texts]
X_test_texts = [singal_word_removal(text) for text in X_test_texts]

In [43]:
total_words_xtrain = 0
for text in X_train_texts:
  total_words_xtrain += len(text.split())

print(f"Total words in X_train_texts: {total_words_xtrain}")

Total words in X_train_texts: 947943
